# Inspect New Omani TTS Dataset

Loads the `dataset_new_omani` corpus via `NewOmaniTTSDataset`, inspects mel
spectrograms, and plays the resampled audio inline.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'commons' / 'dataset.py').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET_ROOT = ROOT / 'dataset_new_omani'
print('Project root :', ROOT)
print('Dataset root :', DATASET_ROOT)
assert DATASET_ROOT.exists(), f'Dataset folder not found: {DATASET_ROOT}'

## 1. Load the dataset

In [ ]:
from commons.dataset import NewOmaniTTSDataset

dataset = NewOmaniTTSDataset(
    dataset_root=str(DATASET_ROOT),
    sample_rate=22050,
    n_fft=1024,
    window_size=1024,
    hop_size=256,
    fmin=0,
    fmax=8000,
    num_mels=80,
)

print(f'Total samples : {len(dataset)}')
print(f'Recording IDs : {dataset.available_speakers}')

## 2. Dataset overview

In [ ]:
import pandas as pd

meta = dataset.metadata.copy()

print('Samples per recording ID:')
print(meta['speaker'].value_counts().to_string())
print()
meta[['File Name', 'Text', 'speaker', 'file_path']].head(8)

## 3. Inspect individual samples

Shows the transcript, waveform, mel spectrogram, and plays the audio.
Change `INDICES` to inspect any samples you like.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from IPython.display import Audio, display
from commons.dataset import load_wav

SR = 22050
SILENCE_SAMPLES = int(0.1 * SR)  # 100 ms (matches NewOmaniTTSDataset)

def show_sample(idx: int):
    row = dataset.metadata.iloc[idx]
    path = row['file_path']
    transcript = row['Text']

    audio = load_wav(path, sr=SR)
    silence = torch.zeros(SILENCE_SAMPLES, dtype=audio.dtype)
    audio = torch.cat([silence, audio, silence])

    transcript_str, mel = dataset[idx]  # mel is (n_mels, T), normalised

    print(f'─── Sample {idx} ───')
    print(f'  File       : {Path(path).name}')
    print(f'  Recording  : {row["speaker"]}')
    print(f'  Audio len  : {len(audio)/SR:.2f}s  ({len(audio):,} samples)')
    print(f'  Mel shape  : {tuple(mel.shape)}  (n_mels × frames)')
    print(f'  Mel range  : [{mel.min():.3f}, {mel.max():.3f}]')
    print(f'  Transcript : {transcript}')

    fig, axes = plt.subplots(2, 1, figsize=(12, 5),
                              gridspec_kw={'height_ratios': [1, 2]})

    t = np.arange(len(audio)) / SR
    axes[0].plot(t, audio.numpy(), lw=0.4, color='steelblue')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title(f'Waveform — {Path(path).name}')
    axes[0].set_xlim(0, t[-1])

    im = axes[1].imshow(mel.numpy(), aspect='auto', origin='lower',
                         interpolation='nearest', cmap='magma')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('Mel bin')
    axes[1].set_title('Normalised Mel Spectrogram')
    plt.colorbar(im, ax=axes[1], label='Normalised dB')

    plt.tight_layout()
    plt.show()

    display(Audio(audio.numpy(), rate=SR))
    print()

In [ ]:
INDICES = [0, 1, 2, 3, 4]

for i in INDICES:
    show_sample(i)

## 4. Sample from each recording ID

In [ ]:
for rec_id in sorted(meta['speaker'].unique()):
    idx = meta[meta['speaker'] == rec_id].index[0]
    print(f'\n=== Recording ID {rec_id} ===')
    show_sample(idx)

## 5. Collator smoke-test

Verifies that `TTSCollator` produces correctly shaped tensors for a small batch.

In [ ]:
from torch.utils.data import DataLoader, random_split
from commons.dataset import TTSCollator

BATCH_SIZE = 4
VAL_SPLIT  = 0.1

n_val   = max(1, int(len(dataset) * VAL_SPLIT))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
print(f'Train / val : {n_train} / {n_val}')

loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=TTSCollator(),
    num_workers=0,
)

text_padded, input_lengths, mel_padded, gate_padded, enc_mask, dec_mask = next(iter(loader))

print('text_padded   :', tuple(text_padded.shape),   '— (B, T_text)')
print('input_lengths :', tuple(input_lengths.shape),  '—', input_lengths.tolist())
print('mel_padded    :', tuple(mel_padded.shape),     '— (B, T_mel, n_mels)')
print('gate_padded   :', tuple(gate_padded.shape),    '— (B, T_mel)')
print('enc_mask      :', tuple(enc_mask.shape))
print('dec_mask      :', tuple(dec_mask.shape))
print('\nAll shapes look correct — dataset is ready for Tacotron2 fine-tuning.')